# Lección 4.3 — Umbrales, curva ROC/AUC y curva Precisión-*Recall*

> **Módulo 4 · Evaluación: ¿mi modelo es bueno?** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 4.2 (precisión, *recall* y matriz de confusión)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar por qué un clasificador en realidad da **probabilidades** y el "sí/no" sale de un **umbral**.
- Predecir qué les pasa a la precisión y al *recall* cuando subes o bajas el umbral.
- Leer una **curva ROC** y explicar el **AUC** con una frase sencilla.
- Leer una **curva Precisión-*Recall*** y saber cuándo es más honesta que la ROC.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             roc_curve, roc_auc_score, precision_recall_curve,
                             average_precision_score)

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: la perilla de sensibilidad del detector de metales

En el aeropuerto, el arco detector de metales tiene una **perilla de sensibilidad**:

```
Sensibilidad ALTA  → suena con un clip o un botón 🔔🔔🔔   (casi nada se le escapa, pero muchas falsas alarmas)
Sensibilidad BAJA  → solo suena con algo grande  🔕        (pocas falsas alarmas, pero se le escapan cosas)
```

El detector **no cambia**: el mismo aparato mide la misma cantidad de metal. Lo que cambia es **a partir de cuánto metal decide sonar**.

Un clasificador funciona igual: por dentro calcula un **puntaje** (una probabilidad) y luego lo compara con un **umbral** para decir "sí" o "no". Ese umbral es una perilla que **tú** eliges.

### ✋ Ejercicio 1 — Calentamiento
¿En cuál de estos lugares pondrías la sensibilidad del detector **más alta**? ¿Por qué?
- a) La entrada de un concierto, con 20,000 personas que quieren entrar rápido.
- b) El acceso a la cabina de un avión.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**b)** En la cabina de un avión, dejar pasar un arma es carísimo; revisar a mano a un piloto por una falsa alarma cuesta poco. En el concierto, una sensibilidad altísima haría filas eternas por cada cinturón.

Esta es la idea de toda la lección: **la sensibilidad correcta no es una propiedad del aparato, sino del costo de cada tipo de error.** (Lo harás con números en la Lección 4.8.)
</details>

---
## 2. 📖 El concepto: probabilidad + umbral = decisión

Un gimnasio quiere detectar a los socios que **van a abandonar** en los próximos tres meses, para ofrecerles una sesión gratis con un entrenador. Tiene datos de 3,000 socios (visitas al mes, antigüedad, pagos…; aquí las columnas son anónimas) y una etiqueta: `1` = abandonó, `0` = se quedó.

Entrenamos un clasificador (una **regresión logística**: la verás a fondo en la Lección 5.2; por ahora basta saber que da una probabilidad). En lugar de pedirle "sí/no", le pedimos la **probabilidad** con `predict_proba`.

In [ ]:
# ▶️ Datos ficticios del gimnasio (≈ 1 de cada 5 socios abandona)
X, y = make_classification(n_samples=3000, n_features=8, n_informative=4, n_redundant=2,
                           weights=[0.8], class_sep=0.8, flip_y=0.02, random_state=3)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

modelo = LogisticRegression().fit(X_tr, y_tr)
prob_te = modelo.predict_proba(X_te)[:, 1]     # probabilidad de "abandona" para cada socio de prueba

pd.DataFrame({
    "prob_abandono": prob_te[:8].round(2),
    "decision_con_0.5": modelo.predict(X_te[:8]),  # lo que dice predict()
    "abandono_real": y_te[:8],
})

Fíjate: `predict()` es simplemente **"¿la probabilidad es ≥ 0.5?"**. Ese 0.5 es una **convención**, no una ley: nadie lo eligió pensando en tu negocio.

### 🔮 Ejercicio 2 — ¿Cuántos detecta?
En el conjunto de prueba hay 900 socios, de los cuales **186** abandonaron de verdad. Con el umbral 0.5:
1. ¿Cuántos socios crees que el modelo marcará como "se va"? ¿Más o menos de 186?
2. ¿Qué *recall* esperas (de los que se fueron, qué porcentaje detecta)?

✍️ *Mis predicciones:*

In [ ]:
marcados = prob_te >= 0.5
print(f"Socios que realmente abandonaron : {y_te.sum()}")
print(f"Socios marcados como 'se va'     : {marcados.sum()}")
print(f"De ellos, abandonaron de verdad  : {(marcados & (y_te == 1)).sum()}")
print(f"Recall con umbral 0.5            : {recall_score(y_te, marcados):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Solo marca a **78** socios, y acierta con **53** de los 186 que se fueron: un *recall* de apenas **28 %**. Con datos desbalanceados (pocos positivos), las probabilidades rara vez superan 0.5, así que el umbral por defecto es **tímido**.

¿Es un mal modelo? No necesariamente. Puede que el modelo **ordene** bien a los socios (los que se van tienen probabilidades más altas) y que el problema sea **dónde pusimos el corte**. Vamos a girar la perilla.
</details>

---
## 3. 📖 Cómo funciona: girar la perilla del umbral

La regla es siempre la misma: **si la probabilidad ≥ umbral → "se va"**. La función de abajo aplica un umbral y te muestra la matriz de confusión (Lección 4.2) desglosada en palabras.

In [ ]:
def evaluar_umbral(umbral):
    pred = (prob_te >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn)
    print(f"Umbral = {umbral}")
    print(f"  Socios marcados como 'se va' : {tp + fp}")
    print(f"  ✅ Detectados (TP)           : {tp}   (se iban y los avisamos)")
    print(f"  🔔 Falsas alarmas (FP)       : {fp}   (se iban a quedar y les regalamos la sesión)")
    print(f"  😶 Se nos escaparon (FN)     : {fn}   (se fueron sin que los viéramos)")
    print(f"  Precisión = {precision:.0%}   ·   Recall = {recall:.0%}")

evaluar_umbral(0.5)

### ✏️ Ejercicio 3 — Gira la perilla
Antes de ejecutar, predice: si **bajas** el umbral a 0.2, ¿qué pasa con el número de socios marcados, la precisión y el *recall*? ¿Y si lo **subes** a 0.7?

✍️ *Mis predicciones:* umbral 0.2 → … · umbral 0.7 → …

Luego cambia `UMBRAL` y ejecuta varias veces (prueba 0.2, 0.7 y 0.1).

In [ ]:
UMBRAL = 0.2    # ✏️ prueba 0.2, 0.7, 0.1 ...
evaluar_umbral(UMBRAL)

<details><summary><b>👀 Ver respuesta</b></summary>

| Umbral | Marcados | Precisión | Recall |
|---|---|---|---|
| 0.1 | 550 | 31 % | 92 % |
| 0.2 | 301 | 46 % | 74 % |
| 0.5 | 78 | 68 % | 28 % |
| 0.7 | 24 | 75 % | 10 % |

- **Bajar el umbral** → el modelo "suena" más: sube el *recall* (se escapan menos) pero baja la precisión (más falsas alarmas).
- **Subir el umbral** → el modelo es más exigente: sube la precisión pero baja el *recall*.

Es un **subibaja**: con el mismo modelo, no puedes subir las dos a la vez solo moviendo el umbral.
</details>

### 📖 La perilla en una tabla

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| **Umbral de decisión** | A partir de qué probabilidad decimos "sí" | Menos alarmas: ⬆️ precisión, ⬇️ *recall* | Más alarmas: ⬆️ *recall*, ⬇️ precisión | 0.5 por convención; el bueno se elige según el costo de cada error (Lección 4.8), con datos de **validación** |

> ⚠️ El umbral **no cambia el modelo**: no hay que reentrenar nada. Solo cambia cómo traducimos la probabilidad en una decisión.

Veamos el subibaja completo, para todos los umbrales a la vez:

In [ ]:
precisiones, recalls, umbrales = precision_recall_curve(y_te, prob_te)
plt.plot(umbrales, precisiones[:-1], label="Precisión")
plt.plot(umbrales, recalls[:-1], label="Recall")
plt.axvline(0.5, color="gray", ls="--", lw=1); plt.text(0.51, 0.05, "umbral por defecto", color="gray")
plt.xlabel("Umbral"); plt.ylabel("Valor"); plt.legend(); plt.grid(alpha=.3)
plt.title("El subibaja precisión–recall"); plt.show()

### ✋ Ejercicio 4 — ¿Dónde pondrías la perilla?
1. La oferta para retener socios es **barata** (una sesión gratis con un entrenador que ya está en nómina). ¿Umbral bajo o alto?
2. Ahora la oferta es **cara**: tres meses gratis. ¿Cambia tu respuesta?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Bajo** (p. ej. 0.2). Regalar una sesión a alguien que se iba a quedar cuesta poco, y perder a un socio cuesta mucho: preferimos *recall* alto aunque haya falsas alarmas.
2. **Más alto.** Ahora cada falsa alarma cuesta tres mensualidades; conviene ser más exigente y ganar precisión.

Fíjate en que **el modelo es el mismo** en los dos casos: solo cambió el negocio. En la Lección 4.8 calcularás el umbral exacto a partir de los costos.
</details>

---
## 4. 📖 La curva ROC: todos los umbrales en un solo dibujo

Probar umbrales uno por uno es lento. La **curva ROC** (*Receiver Operating Characteristic*, un nombre heredado de los radares de la Segunda Guerra Mundial) los prueba **todos** y dibuja, para cada uno, un punto:

- **Eje vertical — Tasa de verdaderos positivos (TPR):** de los que abandonaron, ¿qué fracción detecto? Es el *recall* de siempre.
- **Eje horizontal — Tasa de falsos positivos (FPR):** de los que se quedaron, ¿a qué fracción le doy una falsa alarma?

```
TPR 1 ┤        ●●●●●  ← modelo bueno: sube rápido (detecta mucho con pocas falsas alarmas)
      │     ●●     ·
      │   ●     ·        · · · diagonal = modelo que adivina al azar
      │  ●   ·
      │ ● ·
    0 ┼·───────────── FPR
      0              1
```
Umbral **muy alto** → nadie marcado → esquina (0, 0). Umbral **muy bajo** → todos marcados → esquina (1, 1).

In [ ]:
fpr, tpr, _ = roc_curve(y_te, prob_te)
auc = roc_auc_score(y_te, prob_te)

plt.figure(figsize=(5.5, 5.5))
plt.plot(fpr, tpr, label=f"Nuestro modelo (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], "k:", label="Adivinar al azar (AUC = 0.5)")
for u in [0.1, 0.2, 0.5, 0.7]:            # marcamos algunos umbrales concretos
    pred = prob_te >= u
    x_u, y_u = (pred & (y_te == 0)).sum() / (y_te == 0).sum(), (pred & (y_te == 1)).sum() / (y_te == 1).sum()
    plt.plot(x_u, y_u, "o", color="crimson"); plt.annotate(f" {u}", (x_u, y_u))
plt.xlabel("Tasa de falsos positivos (FPR)"); plt.ylabel("Tasa de verdaderos positivos (TPR = recall)")
plt.legend(loc="lower right"); plt.title("Curva ROC del gimnasio"); plt.show()

### 🔮 Ejercicio 5 — Ubica los umbrales
Sin mirar los puntos rojos con detalle: en la tabla del Ejercicio 3, el umbral 0.1 detecta el 92 % de los que se van. Hay 714 socios que se quedan, y con ese umbral 379 reciben falsa alarma.
1. ¿En qué coordenadas (FPR, TPR) debería estar el punto del umbral 0.1?
2. ¿Hacia qué esquina se mueve el punto si subes el umbral?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. FPR = 379 / 714 ≈ **0.53** y TPR = **0.92**: arriba y a la mitad del eje horizontal. Coincide con el punto rojo "0.1".
2. Hacia **(0, 0)**, abajo a la izquierda: cuanto más exigente el umbral, menos alarmas de ambos tipos. El 0.5 queda en FPR ≈ 0.04, TPR ≈ 0.28.

**Cada punto de la curva es un umbral distinto del mismo modelo.**
</details>

### 📖 El AUC: el área bajo la curva

El **AUC** (*Area Under the Curve*) resume toda la curva en un número entre 0 y 1. Tiene una interpretación preciosa, sin fórmulas:

> **AUC = probabilidad de que, si tomas al azar un socio que abandonó y uno que se quedó, el modelo le dé más probabilidad de abandono al que de verdad abandonó.**

Es decir, el AUC mide qué tan bien **ordena** el modelo, sin importar el umbral. Comprobémoslo comparando **todas** las parejas posibles (186 × 714 ≈ 133,000):

In [ ]:
prob_se_fue = prob_te[y_te == 1]
prob_se_quedo = prob_te[y_te == 0]

# Para cada pareja (uno que se fue, uno que se quedó): ¿el que se fue tiene más probabilidad?
bien_ordenadas = (prob_se_fue[:, None] > prob_se_quedo[None, :]).mean()

print(f"Parejas bien ordenadas : {bien_ordenadas:.3f}")
print(f"AUC de scikit-learn    : {roc_auc_score(y_te, prob_te):.3f}")

### 🔮 Ejercicio 6 — Tres modelos extremos
Predice el AUC de cada uno antes de ejecutar:
1. Un modelo que asigna probabilidades **al azar** (lanza una "moneda").
2. Un modelo **perfecto**, que les da 1 a todos los que se van y 0 a los que se quedan.
3. Un modelo **al revés**: usa `1 − probabilidad` de nuestro modelo.

✍️ *Mis predicciones:* 1 → … · 2 → … · 3 → …

In [ ]:
rng = np.random.default_rng(0)
print(f"1. Al azar   : {roc_auc_score(y_te, rng.random(len(y_te))):.3f}")
print(f"2. Perfecto  : {roc_auc_score(y_te, y_te):.3f}")
print(f"3. Al revés  : {roc_auc_score(y_te, 1 - prob_te):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **≈ 0.5** (aquí 0.463): ordenar al azar acierta la mitad de las parejas. Es la diagonal punteada. Con otra semilla saldría un poco distinto, siempre cerca de 0.5.
2. **1.0**: ordena perfecto todas las parejas.
3. **0.171 = 1 − 0.829**: un AUC por debajo de 0.5 significa que el modelo ordena *al revés*. Tiene información, pero alguien invirtió las etiquetas.

Guía rápida: 0.5 = moneda · 0.7 = algo útil · 0.8 = bueno · 0.9+ = muy bueno (o sospechoso de **fuga de datos**, Lección 2.6).
</details>

---
## 5. 📖 La curva Precisión-*Recall*: la vista desde los positivos

La curva ROC mira las falsas alarmas **en proporción a todos los negativos**. Pero en el día a día, el gerente del gimnasio pregunta otra cosa: *"De los socios que me marcas, ¿cuántos de verdad se iban a ir?"*. Eso es la **precisión**.

La **curva PR** dibuja, para cada umbral, el ***recall*** (eje horizontal) contra la **precisión** (eje vertical). Su resumen es la **precisión promedio** (AP, *Average Precision*), también entre 0 y 1.

Una diferencia clave: en la ROC, el modelo al azar es siempre la diagonal (AUC 0.5). En la PR, **el modelo al azar es una línea horizontal a la altura de la prevalencia** (el porcentaje de positivos).

In [ ]:
ap = average_precision_score(y_te, prob_te)
prevalencia = y_te.mean()

plt.plot(recalls, precisiones, label=f"Nuestro modelo (AP = {ap:.2f})")
plt.axhline(prevalencia, color="k", ls=":", label=f"Adivinar al azar (= prevalencia {prevalencia:.2f})")
plt.xlabel("Recall"); plt.ylabel("Precisión"); plt.ylim(0, 1.02)
plt.legend(); plt.title("Curva Precisión-Recall del gimnasio"); plt.show()

### ✋ Ejercicio 7 — La línea del azar
¿Por qué un modelo que marca socios **al azar** tiene una precisión de ≈ 0.21, sin importar cuántos marque?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Porque si eliges socios al azar, la proporción de los que se van entre los marcados será la misma que en toda la población: **20.7 %**. Marcar más o menos socios al azar cambia el *recall*, pero no la precisión.

Por eso un AP de 0.57 hay que leerlo **comparado con 0.21**: el modelo es casi 3 veces más preciso que el azar. Un AP de 0.57 sería pésimo si la prevalencia fuera 0.50.
</details>

---
## 6. 📖 ¿Qué pasa si los positivos son muy raros?

Ahora la pregunta típica de entrevista: ¿qué curva te dice la verdad cuando los positivos son **muy** escasos?

Vamos a comparar dos situaciones con el mismo tipo de datos y el mismo tipo de modelo:
- **Negocio A:** 20 % de positivos.
- **Negocio B:** 1 % de positivos (como una enfermedad rara o un defecto de fábrica).

### 🔮 Ejercicio 8 — ¿Qué número se derrumba?
De A a B, ¿qué crees que le pasa al **AUC de la ROC**? ¿Y a la **precisión promedio (AP)**? ¿Bajan los dos, ninguno, solo uno?

✍️ *Mi predicción:*

In [ ]:
for nombre, peso in [("A (20 % positivos)", 0.80), ("B ( 1 % positivos)", 0.99)]:
    Xd, yd = make_classification(n_samples=20_000, n_features=8, n_informative=4, n_redundant=2,
                                 weights=[peso], class_sep=0.8, flip_y=0, random_state=3)
    Xa, Xb, ya, yb = train_test_split(Xd, yd, test_size=0.5, stratify=yd, random_state=0)
    p = LogisticRegression().fit(Xa, ya).predict_proba(Xb)[:, 1]
    print(f"Negocio {nombre}:  AUC-ROC = {roc_auc_score(yb, p):.2f}   ·   "
          f"AP = {average_precision_score(yb, p):.2f}   (azar en PR = {yb.mean():.2f})")

<details><summary><b>👀 Ver respuesta</b></summary>

- El **AUC-ROC casi no cambia** (0.84 → 0.83): el modelo **ordena** igual de bien en los dos negocios.
- La **AP se derrumba** (0.44 → 0.03): en el negocio B, aunque el modelo ordena bien, entre las alarmas hay **muchísimos** negativos, simplemente porque hay 99 negativos por cada positivo. Una FPR "pequeña" del 5 % sobre 9,900 negativos son ~500 falsas alarmas para encontrar, como mucho, 100 positivos. Y ojo: 0.03 sigue siendo mejor que el azar (0.01), pero, en promedio, de cada 100 alarmas unas 97 serían falsas.

Moraleja: **con positivos muy raros, la ROC se ve optimista; la curva PR muestra el dolor real** (cuántas alarmas falsas tendrá que revisar alguien). Si te importan las alarmas, mira la PR.
</details>

### 📖 ¿Cuál uso?

| Situación | Mira… | Por qué |
|---|---|---|
| Clases más o menos balanceadas, te importa **ordenar** bien | Curva ROC / AUC | Resume bien y es fácil de comparar entre modelos |
| Positivos **raros** (< 10 %) y te importa la calidad de las alarmas | Curva PR / AP | La ROC esconde la avalancha de falsos positivos |
| Tienes que **tomar decisiones** (aprobar, llamar, revisar) | Un **umbral concreto** + precisión, *recall* y costo | El AUC y la AP no eligen el umbral por ti (Lección 4.8) |

> 💡 El AUC y la AP **no dependen del umbral**: resumen *todos* los umbrales. Por eso sirven para comparar modelos, pero **no** te dicen dónde poner la perilla.

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| El modelo casi nunca dice "sí" con datos desbalanceados | Umbral 0.5 por defecto | Bajar el umbral, mirando precisión y *recall* |
| Elegiste el umbral mirando el conjunto de **prueba** | Ajustar con prueba = hacer trampa | Elegir el umbral con validación (Lección 4.6) y probar una sola vez |
| AUC 0.95 pero el equipo se queja de "demasiadas alarmas" | Positivos raros: la ROC se ve optimista | Mirar la curva PR y la precisión al umbral elegido |
| Comparas dos modelos por exactitud a umbral 0.5 | Uno puede ser mejor a otro umbral | Comparar AUC/AP, y luego decidir el umbral |
| AUC por debajo de 0.5 | Etiquetas o probabilidades invertidas | Revisar qué columna de `predict_proba` usas (`[:, 1]` = clase 1) |
| Tratas la probabilidad como "certeza real" (p. ej. "70 % = 7 de cada 10") | Las probabilidades de muchos modelos no están **calibradas** | Verificarlo con una curva de calibración (`CalibrationDisplay` en scikit-learn) |

### 🔮 Ejercicio 9 — El reporte sospechoso
Un compañero te escribe: *"¡Mi modelo para detectar piezas defectuosas tiene AUC de 0.96! Ya lo podemos poner en la fábrica."* Solo el 0.5 % de las piezas son defectuosas.

¿Qué dos preguntas le harías antes de celebrar?

✍️ *Mis preguntas:*

<details><summary><b>👀 Ver respuesta</b></summary>

Por ejemplo:
1. **"¿Cuál es la curva PR / la AP?"** Con 0.5 % de defectos, un AUC alto puede convivir con una precisión bajísima: quizá de cada 10 piezas que el modelo marca, solo 1 es defectuosa.
2. **"¿Qué umbral vas a usar y cuántas alarmas por día generará?"** El AUC no elige el umbral. Hay que traducirlo a algo operativo: *"de cada 1,000 piezas, paramos 30 y encontramos 4 de las 5 defectuosas"*.

Bonus: *"¿Lo mediste con datos que el modelo no vio, y sin tocar el conjunto de prueba para ajustar?"* Un 0.96 también puede ser una señal de fuga de datos (Lección 2.6).
</details>

---
## 8. 🧠 Resumen en 5 puntos
1. Un clasificador da **probabilidades**; el "sí/no" sale de compararlas con un **umbral**, y el 0.5 es solo una convención.
2. Bajar el umbral sube el ***recall*** y baja la **precisión**; subirlo hace lo contrario. Es un **subibaja**.
3. La **curva ROC** dibuja todos los umbrales (TPR vs. FPR). Su **AUC** es la probabilidad de ordenar bien una pareja positivo–negativo: 0.5 = azar, 1 = perfecto.
4. La **curva PR** (precisión vs. *recall*) y su **AP** son más honestas cuando los positivos son **raros**; su línea del azar es la **prevalencia**.
5. AUC y AP **comparan modelos** pero **no eligen el umbral**: eso depende del costo de cada error (Lección 4.8) y se decide con datos de validación.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** Bajas el umbral de 0.5 a 0.3. ¿Qué es lo **más** probable?
- a) Suben la precisión y el *recall*
- b) Sube el *recall* y baja la precisión
- c) Sube la precisión y baja el *recall*
- d) No cambia nada porque el modelo es el mismo

**2.** Un modelo tiene AUC = 0.5. Significa que…
- a) Acierta el 50 % de las veces
- b) Ordena a positivos y negativos igual que una moneda
- c) Tiene una precisión del 50 %
- d) Su umbral óptimo es 0.5

**3.** Hay 2 % de positivos. Un modelo al azar en la curva PR tiene una precisión de…
- a) 0.5 · b) 0.02 · c) 0.98 · d) Depende del umbral

**4.** ¿Cuál de estas cosas **requiere** reentrenar el modelo?
- a) Cambiar el umbral de 0.5 a 0.2
- b) Dibujar la curva ROC
- c) Calcular la AP
- d) Ninguna de las anteriores

**5.** El modelo A tiene AUC 0.85 y el modelo B, 0.80. ¿Qué puedes afirmar con seguridad?
- a) A tendrá mejor precisión que B en cualquier umbral
- b) En promedio, A ordena mejor a los positivos por encima de los negativos
- c) A debe usar umbral 0.5
- d) A tiene mayor exactitud

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Más socios superan el umbral: se escapan menos positivos (*recall* ⬆️) pero entran más falsas alarmas (precisión ⬇️).
2. **b)** El AUC mide el orden, no el porcentaje de aciertos. Un modelo con AUC 0.5 puede tener 98 % de exactitud si siempre dice "no" con 2 % de positivos.
3. **b)** La línea del azar en la curva PR es la prevalencia.
4. **d)** Ninguna. Todas usan las mismas probabilidades del modelo ya entrenado.
5. **b)** El AUC es un promedio sobre todos los umbrales; las curvas pueden cruzarse, y en *tu* umbral B podría ser mejor.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Dos modelos tienen el mismo AUC, pero sus curvas ROC se cruzan: A es mejor a la izquierda (pocas falsas alarmas) y B a la derecha (*recall* muy alto). Si vas a usar el modelo para un **tamizaje médico** en el que no se puede escapar casi ningún enfermo, ¿cuál eliges?

**B. Caso de análisis 💼** Un banco revisa a mano cada transferencia que el modelo marca como sospechosa. El equipo de revisión solo puede revisar **200 transferencias al día** de 50,000. El jefe pregunta: *"¿Qué umbral uso?"*. ¿Cómo usarías esa restricción para elegir el umbral? ¿Qué métrica te importa más en esas 200?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** **B.** En el tamizaje trabajarás en la zona de *recall* altísimo (parte derecha de la curva), y ahí B es mejor. El AUC igual esconde esa diferencia: **mira la zona de la curva que vas a usar**, no solo el área.

**B.** La restricción **fija el umbral por ti**: ordenas las 50,000 por probabilidad y marcas las 200 más altas (equivale a poner el umbral en la probabilidad de la transferencia número 200). Lo que importa es la **precisión en esas 200** (cuántas son de verdad sospechosas: *precision@200*) y qué fracción de todos los casos reales capturas (*recall*). Un modelo con mejor AP suele ganar aquí, porque la AP premia tener positivos arriba de la lista.
</details>

### 9.3 🧩 Práctica integradora: el examen de tamizaje

Una clínica quiere usar un modelo como **examen de tamizaje** (*screening*) para una enfermedad que tiene ~10 % de los pacientes. A quien el modelo marque se le hace un estudio más caro y confirmatorio. La directora médica pone una condición:

> 🩺 **"El tamizaje debe detectar al menos al 90 % de los enfermos."**

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Datos y tres conjuntos.** Separamos **entrenamiento** (para entrenar), **validación** (para elegir el umbral) y **prueba** (para el examen final, una sola vez).

In [ ]:
Xm, ym = make_classification(n_samples=5000, n_features=10, n_informative=5, n_redundant=2,
                             weights=[0.9], class_sep=0.8, n_clusters_per_class=1,
                             flip_y=0.01, random_state=11)
Xm_resto, Xm_te, ym_resto, ym_te = train_test_split(Xm, ym, test_size=0.25, stratify=ym, random_state=0)
Xm_tr, Xm_va, ym_tr, ym_va = train_test_split(Xm_resto, ym_resto, test_size=1/3, stratify=ym_resto, random_state=0)

print(f"Entrenamiento: {len(ym_tr)} · Validación: {len(ym_va)} ({ym_va.sum()} enfermos) · Prueba: {len(ym_te)} ({ym_te.sum()} enfermos)")
print(f"Prevalencia: {ym.mean():.1%}")

clinica = LogisticRegression().fit(Xm_tr, ym_tr)
prob_va = clinica.predict_proba(Xm_va)[:, 1]

**Paso 2 🔮 — ¿Cumple con el umbral por defecto?** Predice: con umbral 0.5, ¿el modelo cumple la condición del 90 %?

✍️ *Mi apuesta:*

In [ ]:
pred_va = prob_va >= 0.5
print(f"Umbral 0.5 → recall {recall_score(ym_va, pred_va):.1%} · precisión {precision_score(ym_va, pred_va):.1%} · marcados {pred_va.sum()}")

<details><summary><b>👀 Ver respuesta</b></summary>

**No cumple**: *recall* de **75 %** (se escapa 1 de cada 4 enfermos), aunque con una precisión altísima (91 %). El umbral 0.5 es demasiado tímido para un tamizaje.
</details>

**Paso 3 ✋ — ¿El modelo ordena bien?** Antes de mover el umbral, revisa si vale la pena: calcula el AUC y la AP en validación, y compara la AP con la prevalencia.

In [ ]:
print(f"AUC-ROC: {roc_auc_score(ym_va, prob_va):.3f}")
print(f"AP     : {average_precision_score(ym_va, prob_va):.3f}   (azar = {ym_va.mean():.3f})")

<details><summary><b>👀 Ver respuesta</b></summary>

AUC ≈ **0.97** y AP ≈ **0.90**, frente a un azar de 0.10. El modelo **ordena muy bien**: el problema del paso 2 no es el modelo, es el umbral. Vale la pena girar la perilla.
</details>

**Paso 4 ✏️ — Busca el umbral.** Usando **solo validación**, busca el umbral **más alto** que todavía detecte al menos el 90 % de los enfermos (el más alto, para no generar más falsas alarmas de las necesarias).

In [ ]:
RECALL_OBJETIVO = 0.90     # ✏️ prueba 0.95 o 0.80

prec_va, rec_va, umb_va = precision_recall_curve(ym_va, prob_va)
umbral_elegido = umb_va[rec_va[:-1] >= RECALL_OBJETIVO].max()

pred_va = prob_va >= umbral_elegido
print(f"Umbral elegido: {umbral_elegido:.3f}")
print(f"Validación → recall {recall_score(ym_va, pred_va):.1%} · precisión {precision_score(ym_va, pred_va):.1%} · marcados {pred_va.sum()} de {len(ym_va)}")

<details><summary><b>👀 Ver respuesta</b></summary>

El umbral queda en ≈ **0.18** (muy lejos del 0.5). En validación detecta al **90 %** de los enfermos con una precisión de **70 %**: de cada 10 pacientes que mandamos al estudio confirmatorio, unos 7 están enfermos de verdad. Marca a 168 pacientes de 1,250.

✏️ Si pides 95 %, el umbral baja y la precisión cae: cada punto extra de *recall* cuesta cada vez más falsas alarmas.
</details>

**Paso 5 ▶️ — Examen final, una sola vez.** Aplicamos **el mismo umbral** (sin volver a elegirlo) al conjunto de prueba.

In [ ]:
prob_te_m = clinica.predict_proba(Xm_te)[:, 1]
pred_te_m = prob_te_m >= umbral_elegido
tn, fp, fn, tp = confusion_matrix(ym_te, pred_te_m).ravel()
print(f"Prueba → recall {recall_score(ym_te, pred_te_m):.1%} · precisión {precision_score(ym_te, pred_te_m):.1%}")
print(f"Enfermos detectados: {tp} de {tp + fn}  ·  falsas alarmas: {fp}")
print(f"(Con el umbral 0.5 el recall en prueba sería {recall_score(ym_te, prob_te_m >= 0.5):.1%})")

✋ El *recall* en prueba, ¿cumple **exactamente** la condición? ¿Por qué podría no coincidir con validación?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Queda en **89.2 %** (116 de 130 enfermos): **un pelito por debajo** del 90 %. Con solo 130 enfermos en cada conjunto, un paciente más o menos mueve el *recall* casi un punto. El umbral se eligió para que cumpliera *justo* en validación, así que en datos nuevos puede quedar un poco arriba o un poco abajo.

Lección práctica: **deja margen**. Si la condición es 90 %, apunta a ~93 % en validación, o usa validación cruzada (Lección 4.6) para elegir el umbral con más datos. Aun así, compáralo con el **72 %** que habrías tenido con el umbral 0.5.
</details>

**Paso 6 ✋ — Explícaselo a la directora.** Escribe en dos o tres frases, sin jerga, qué hace el tamizaje y qué precio se paga.

✍️ *Mi explicación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Por ejemplo: *"El tamizaje marca a un poco más de 1 de cada 8 pacientes para el estudio confirmatorio. Entre ellos están casi 9 de cada 10 enfermos. El precio: de cada 10 pacientes que mandamos al estudio, unos 3 resultarán sanos. Si queremos detectar todavía más enfermos, tendremos que mandar a muchos más sanos al estudio."*

Fíjate: no dijiste "AUC" ni "umbral". Hablaste de **personas** y **decisiones**.
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, "ROC and AUC, Clearly Explained!" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, secciones *Receiver operating characteristic (ROC)* y *Precision, recall and F-measures*; y `TunedThresholdClassifierCV` para elegir el umbral automáticamente (lo usarás en la Lección 4.8).
- 📄 **Lectura:** Saito y Rehmsmeier (2015), *"The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets"*.

---
⬅️ **Anterior:** [4.2 Métricas de clasificación: más allá de la exactitud](4.2-metricas-de-clasificacion.ipynb) · ➡️ **Siguiente:** [4.4 Sobreajuste vs. subajuste: memorizar no es aprender](4.4-sobreajuste-vs-subajuste.ipynb)